# Markov Decision Processes: Can-Collecting Robot

## Introduction

In this notebook, we model and simulate a can-collecting robot operating in an office environment using a Markov Decision Process (MDP). The robot operates under two possible battery states — high and low — and must decide between searching for cans, waiting, or recharging its battery.

The system is governed by two key parameters: **α** (probability of maintaining a high battery level while searching) and **β** (probability of maintaining a low battery level while searching). These parameters directly influence the robot's behavior, its accumulated reward, and the risk of being rescued after battery depletion.

## Task 1: Simulation of Transition Dynamics

### 1.1 Environment Setup

In [1]:
import random
def simulation_trial(state, action, dynamics):
    options = []
    probabilities = []
    for (s, a, s_next, r), prob in dynamics.items():
        if s == state and a == action:
            options.append((s_next, r))
            probabilities.append(prob)
    if not options:
        raise ValueError(f"No dynamics found for state '{state}' and action '{action}'")
    next_state, reward = random.choices(options, probabilities)[0]
    return next_state, reward

The environment is modeled as a finite MDP with two possible states: **high** and **low**, representing the robot's battery charge level. Depending on the current state, the robot can choose from a set of valid actions — searching and
waiting when the battery is high, and searching, waiting, or recharging when it is low. The system's stochastic behavior is governed by four parameters: **α**, **β**, **r_search**, and **r_wait**, which define the transition probabilities and expected rewards for each state-action pair.

### 1.2 Transition Dynamics

In [2]:
def run_scenario(alpha, beta, r_search=5, r_wait=1, trials=10):
    print(f"\n{'='*60}")
    print(f"  SCENARIO | alpha={alpha}, beta={beta}, r_search={r_search}, r_wait={r_wait}")
    print(f"{'='*60}")

    # Equation p(s',r | s,a) -> Stochastic
    # The tuple (s, a, s_next, r) represents:
    # (s: actual, a: action, s_next: next state, r: reward): probability
    dynamics = {
        # High - Search
        ("high", "search", "high", r_search): alpha,
        ("high", "search", "low",  r_search): (1 - alpha),
        # Low - Search
        ("low", "search", "low",  r_search): beta,
        ("low", "search", "high", -3):       (1 - beta),
        # High - Wait
        ("high", "wait", "high", r_wait): 1,
        # Low - Wait
        ("low", "wait", "low",  r_wait): 1,
        # Low - Recharge
        ("low", "recharge", "high", 0): 1,
    }

    simulations = [
        ("low",  "search"),
        ("high", "search"),
        ("low",  "wait"),
        ("high", "wait"),
        ("low",  "recharge"),
    ]

    for state, action in simulations:
        print(f"\n  --- {state.upper()} - {action.upper()} ---")
        for i in range(1, trials + 1):
            next_state, reward = simulation_trial(state, action, dynamics)
            print(f"    Trial {i}: Next State = {next_state}, Reward = {reward}")

The transition dynamics of the MDP are represented as a Python dictionary, where each key is a tuple **(s, a, s', r)**, current state, action, next state, and reward, and each value is the corresponding transition probability. This
structure directly encodes the equation **p(s', r | s, a)**, which defines the probability of reaching state s' with reward r, given that the agent is in state s and takes action a. The dynamics are stochastic for the search action, where the outcome depends on α and β, while the wait and recharge actions are fully deterministic — they always produce the same next state and reward regardless of the parameters.

### 1.3 Simulation Results

In [3]:
run_scenario(alpha=0.8, beta=0.6)
run_scenario(alpha=0.5, beta=0.4)
run_scenario(alpha=0.3, beta=0.7)


  SCENARIO | alpha=0.8, beta=0.6, r_search=5, r_wait=1

  --- LOW - SEARCH ---
    Trial 1: Next State = high, Reward = -3
    Trial 2: Next State = high, Reward = -3
    Trial 3: Next State = low, Reward = 5
    Trial 4: Next State = high, Reward = -3
    Trial 5: Next State = low, Reward = 5
    Trial 6: Next State = high, Reward = -3
    Trial 7: Next State = low, Reward = 5
    Trial 8: Next State = low, Reward = 5
    Trial 9: Next State = high, Reward = -3
    Trial 10: Next State = low, Reward = 5

  --- HIGH - SEARCH ---
    Trial 1: Next State = high, Reward = 5
    Trial 2: Next State = high, Reward = 5
    Trial 3: Next State = high, Reward = 5
    Trial 4: Next State = high, Reward = 5
    Trial 5: Next State = high, Reward = 5
    Trial 6: Next State = high, Reward = 5
    Trial 7: Next State = high, Reward = 5
    Trial 8: Next State = high, Reward = 5
    Trial 9: Next State = high, Reward = 5
    Trial 10: Next State = low, Reward = 5

  --- LOW - WAIT ---
    Trial 1:

### 1.4 Discussion

The results confirm the expected stochastic behavior of the MDP. In the **high - search** action, the reward was always +5 regardless of the next state, since both transitions share the same reward r_search. This means that searching from a high battery level is always profitable, with the only risk being a possible drop to a low battery state.

In contrast, the **low - search** action showed high sensitivity to the value of β. In Scenario 1 (β=0.6), approximately 50% of trials resulted in a rescue (reward = -3), while in Scenario 2 (β=0.4), 80% of trials ended in rescue confirming that a lower β dramatically increases the risk of battery depletion.
Interestingly, Scenario 3 (β=0.7) showed better survival rates in low - search, demonstrating that a higher β effectively protects the robot even when starting from a low state.

The **wait** and **recharge** actions behaved deterministically across all scenarios, as expected — wait always preserved the current state with reward r_wait, and recharge always recovered the battery to high with no reward. These actions are unaffected by α and β, making them reliable but less rewarding alternatives.

## Task 2: Visualizations

### 2.1 State Frequency per Scenario

In [4]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

def simulation_trial(state, action, dynamics):
    options = []
    probabilities = []

    for (s, a, s_next, r), prob in dynamics.items():
        if s == state and a == action:
            options.append((s_next, r))
            probabilities.append(prob)
    if not options:
        raise ValueError(f"No dynamics found for state '{state}' and action '{action}'")
    next_state, reward = random.choices(options, probabilities)[0]
    return next_state, reward

def run_scenario(alpha, beta, r_search=5, r_wait=1):
    states_over_time = []
    accumulated_reward = []
    print(f"\n{'='*60}")
    print(f"  SCENARIO | alpha={alpha}, beta={beta}, r_search={r_search}, r_wait={r_wait}")
    print(f"{'='*60}")

    # Equation p(s',r | s,a) -> Stochastic
    # The tuple (s, a, s_next, r) represents:
    # (s: actual, a: action, s_next: next state, r: reward): probability
    dynamics = {
        # High - Search
        ("high", "search", "high", r_search): alpha,
        ("high", "search", "low",  r_search): (1 - alpha),
        # Low - Search
        ("low", "search", "low",  r_search): beta,
        ("low", "search", "high", -3):       (1 - beta),
        # High - Wait
        ("high", "wait", "high", r_wait): 1,
        # Low - Wait
        ("low", "wait", "low",  r_wait): 1,
        # Low - Recharge
        ("low", "recharge", "high", 0): 1,
    }

    valid_actions = {
        "high": ["search", "wait"],
        "low":  ["search", "wait", "recharge"]
    }

    for episode in range(50):
        current_state = random.choice(["high", "low"])
        initial_state = current_state  # guardamos para el print
        total_reward = 0

        for step in range(10):
            action = random.choice(valid_actions[current_state])
            next_state, reward = simulation_trial(current_state, action, dynamics)
            total_reward += reward
            current_state = next_state
            states_over_time.append(current_state)

        accumulated_reward.append(total_reward)
        print(f"  Episode {episode + 1:2d} | Initial: {initial_state:4s} | Final: {current_state:4s} | Reward: {total_reward}")
    return states_over_time, accumulated_reward

For this task, the simulation was restructured to support sequential episodes and data collection for visualization. Three key changes were introduced compared to Task 1.

First, the simulation now runs **50 episodes of 10 steps each**, where the next state of each step becomes the current state of the following one — creating a truly sequential simulation instead of independent trials.

Second, actions are now selected **randomly** from the set of valid actions for each state, using `random.choice`. This avoids any policy bias and allows the visualizations to reflect the general behavior of the system rather than a specific strategy.

Third, two data structures are collected during the simulation: **states_over_time**, which records the state at every step across all episodes (500 data points total), and **accumulated_reward**, which stores the total reward obtained per episode (50 data points total). These structures feed directly into the three visualizations presented below.

In [5]:
scenarios = [
    {"alpha": 0.8, "beta": 0.6},
    {"alpha": 0.5, "beta": 0.4},
    {"alpha": 0.3, "beta": 0.7},
    {"alpha": 0.2, "beta": 0.3},
    {"alpha": 0.7, "beta": 0.3}
]

results = []
for scenario in scenarios:
    states, rewards = run_scenario(**scenario)
    results.append({
        "alpha": scenario["alpha"],
        "beta": scenario["beta"],
        "states": states,
        "rewards": rewards
    })

for i in range(len(results)):
    high_count = results[i]["states"].count("high")
    low_count = results[i]["states"].count("low")


  SCENARIO | alpha=0.8, beta=0.6, r_search=5, r_wait=1
  Episode  1 | Initial: low  | Final: low  | Reward: 21
  Episode  2 | Initial: high | Final: high | Reward: 18
  Episode  3 | Initial: high | Final: low  | Reward: 26
  Episode  4 | Initial: low  | Final: low  | Reward: 34
  Episode  5 | Initial: high | Final: high | Reward: 26
  Episode  6 | Initial: low  | Final: low  | Reward: 32
  Episode  7 | Initial: high | Final: high | Reward: 29
  Episode  8 | Initial: low  | Final: high | Reward: 18
  Episode  9 | Initial: low  | Final: low  | Reward: 33
  Episode 10 | Initial: high | Final: low  | Reward: 29
  Episode 11 | Initial: high | Final: high | Reward: 18
  Episode 12 | Initial: high | Final: high | Reward: 26
  Episode 13 | Initial: high | Final: high | Reward: 24
  Episode 14 | Initial: high | Final: high | Reward: 29
  Episode 15 | Initial: low  | Final: high | Reward: 18
  Episode 16 | Initial: high | Final: high | Reward: 21
  Episode 17 | Initial: low  | Final: high | Rew

Five scenarios were defined with different combinations of α and β to capture a wide range of system behaviors — from high stability (α=0.8, β=0.6) to high risk (α=0.2, β=0.3). Each scenario was simulated using `run_scenario` and its results stored in a list of dictionaries containing the states and rewards collected across all episodes. Additionally, the frequency of each battery state was computed by counting the occurrences of **high** and **low** across the 500 recorded steps, providing the foundation for the visualizations presented in the following sections.

In [6]:
fig1 = make_subplots(rows=1, cols=5, subplot_titles=[
    f"α={r['alpha']}, β={r['beta']}" for r in results
])

for i, result in enumerate(results):
    high_count = result["states"].count("high")
    low_count  = result["states"].count("low")

    fig1.add_trace(
        go.Bar(x=["High", "Low"], y=[high_count, low_count],
               marker_color=["blue", "orange"], showlegend=False),
        row=1, col=i+1
    )

fig1.update_layout(title_text="State Frequency per Scenario")
fig1.show()

The state frequency chart reveals a consistent pattern across all five scenarios: the robot spends significantly more time in the **high** battery state than in the **low** state. This is largely explained by the random action selection when in a low state, the recharge action always transitions back to high, naturally pushing the system toward the high state more frequently.

The most notable difference between scenarios is the gap between high and low frequencies. Scenario 1 (α=0.8, β=0.6) and Scenario 5 (α=0.7, β=0.3) shows the largest gap, with
398 steps in high versus only 102 in low, reflecting the strong probability of maintaining a high battery level during search. In contrast, Scenario 3 (α=0.3, β=0.7) shows the most balanced distribution — 273 high versus 227 low — since a low α means the robot frequently drops to a low state while
searching from high.

Interestingly, Scenario 4 (α=0.2, β=0.3) also shows a relatively balanced distribution despite having the lowest α, because the low β increases rescues which reset the battery back to high, partially compensating for the frequent drops.

### 2.2 Reward per Episode and Accumulated Reward

In [7]:
fig2 = make_subplots(rows=1, cols=2, subplot_titles=[
    "Reward per Episode", "Accumulated Reward over Time"
])

colors = ["blue", "orange", "green", "red", "purple"]
episodes = list(range(1, 51))

for i, result in enumerate(results):
    alpha   = result["alpha"]
    beta    = result["beta"]
    rewards = result["rewards"]
    label   = f"α={alpha}, β={beta}"

    fig2.add_trace(
        go.Scatter(x=episodes, y=rewards,
                   mode="lines", name=label,
                   line=dict(color=colors[i])),
        row=1, col=1
    )

    fig2.add_trace(
        go.Scatter(x=episodes, y=list(np.cumsum(rewards)),
                   mode="lines", name=label,
                   line=dict(color=colors[i]),
                   showlegend=False),
        row=1, col=2
    )

fig2.update_xaxes(title_text="Episode")
fig2.update_yaxes(title_text="Reward", col=1)
fig2.update_yaxes(title_text="Accumulated Reward", col=2)
fig2.update_layout(title_text="Reward Analysis per Scenario")
fig2.show()


The reward per episode chart shows high variability across all scenarios, which  is expected given the random action selection and the stochastic nature of the transitions. Despite this noise, some patterns emerge — Scenario 1 (α=0.8, β=0.6) consistently maintains higher reward values per episode, rarely dropping below 15, while Scenario 4 (α=0.2, β=0.3) shows the most volatile behavior, with several episodes dropping close to 10, reflecting the combined effect of a low  α (frequent battery drops from high) and a low β (high rescue probability from low).

The accumulated reward chart makes the differences between scenarios much clearer.  Scenario 1 (α=0.8, β=0.6) achieves the highest accumulated reward at approximately 1250 after 50 episodes, confirming that high values of both α and β consistently produce the best long-term performance. Scenario 5 (α=0.7, β=0.3) starts competitively but falls behind around episode 20, showing that a low β eventually penalizes the robot despite a strong α. Scenario 4 (α=0.2, β=0.3) accumulates the least reward — around 900 — demonstrating that low values of both parameters significantly limit the robot's performance over time.

Overall, these results confirm that α has a strong influence on sustained performance, while β primarily affects the risk of large reward penalties through rescues. The best performing scenarios are those that combine a high α to stay
in the high state with a sufficiently high β to avoid rescues when in the low state.

### 2.3 Reward Distribution Comparison

In [8]:
fig3 = go.Figure()

for i, result in enumerate(results):
    label = f"α={result['alpha']}, β={result['beta']}"

    fig3.add_trace(
        go.Box(y=result["rewards"], name=label, boxmean=True)
    )

fig3.update_layout(
    title_text="Reward Distribution per Scenario",
    xaxis_title="Scenario",
    yaxis_title="Reward per Episode"
)
fig3.show()

The box plot provides a clearer view of the reward distribution across scenarios by showing not just the average performance but also the spread and consistency of results.

Scenario 1 (α=0.8, β=0.6) stands out as the best performing scenario, with the highest median at approximately 25 and a relatively compact IQR between 21 and 30. This indicates that not only does the robot collect more reward on average, but it does so consistently across episodes. The mean (dashed line) closely follows the median, suggesting a symmetric and stable distribution.

Scenario 4 (α=0.2, β=0.3) shows the lowest median at 17 and the widest spread among all scenarios, with the whisker reaching down to 9. This confirms that low values of both α and β produce the most unstable and unrewarding behavior. Notably, this scenario also contains the only visible outlier in the chart — a point above 31 — indicating occasional lucky episodes that are far from the typical outcome.

Scenario 5 (α=0.7, β=0.3) presents an interesting case — it has the widest overall range, with the upper whisker reaching 41, but also drops as low as 10. This high variance reflects the tension between a strong α keeping the battery high and a weak β causing frequent rescues when the battery drops to low.

Scenario 3 (α=0.3, β=0.7) achieves a moderate and relatively stable performance, with a compact IQR and median around 21. This suggests that a high β can partially compensate for a low α by protecting the robot from rescues when in the low state.

Overall, the box plot confirms that **α and β must both be high** to achieve consistent and high rewards. Optimizing only one parameter while neglecting the other leads to either high variance or low overall performance.

### 2.4 Discussion

The three visualizations presented in this task provide complementary perspectives on how α and β influence the robot's behavior and overall performance.

From the state frequency analysis, it is clear that the robot tends to spend more time in the high battery state across all scenarios, driven by the recharge action which always recovers the battery to high. However, the balance between high and low states is heavily influenced by α — scenarios with a low α, such as Scenario 3 (α=0.3, β=0.7), show a much more balanced distribution, as the robot frequently drops to low while searching from high.

The reward analysis confirms that higher values of both α and β consistently produce better long-term performance. Scenario 1 (α=0.8, β=0.6) accumulated the highest total reward after 50 episodes, while Scenario 4 (α=0.2, β=0.3) lagged significantly behind. The per-episode reward chart also revealed that low parameter values introduce greater volatility, making the robot's performance harder to predict.

Finally, the reward distribution analysis reinforced these findings by showing that Scenario 1 not only achieves the highest median reward but also the most consistent results, while Scenario 4 produces the widest spread and lowest median.
Scenario 5 (α=0.7, β=0.3) highlighted that optimizing α alone is insufficient a low β introduces high variance that undermines the gains from maintaining a high battery state.

In summary, both parameters play a critical and complementary role: α governs the robot's ability to stay in the high state, while β determines the risk of costly rescues when operating from the low state. The best overall performance
is achieved when both parameters are high, ensuring stable and rewarding behavior across episodes.

## Task 3: Policy Evaluation

### 3.1 Policy Definition

In [9]:
def run_scenario(alpha, beta, policy, r_search=5, r_wait=1):
    states_over_time = []
    accumulated_reward = []
    print(f"\n{'='*60}")
    print(f"  SCENARIO | alpha={alpha}, beta={beta}, r_search={r_search}, r_wait={r_wait}")
    print(f"{'='*60}")

    # Equation p(s',r | s,a) -> Stochastic
    # The tuple (s, a, s_next, r) represents:
    # (s: actual, a: action, s_next: next state, r: reward): probability
    dynamics = {
        # High - Search
        ("high", "search", "high", r_search): alpha,
        ("high", "search", "low",  r_search): (1 - alpha),
        # Low - Search
        ("low", "search", "low",  r_search): beta,
        ("low", "search", "high", -3):       (1 - beta),
        # High - Wait
        ("high", "wait", "high", r_wait): 1,
        # Low - Wait
        ("low", "wait", "low",  r_wait): 1,
        # Low - Recharge
        ("low", "recharge", "high", 0): 1,
    }

    for episode in range(50):
        current_state = random.choice(["high", "low"])
        initial_state = current_state
        total_reward = 0

        for step in range(10):
            action = policy[current_state]
            next_state, reward = simulation_trial(current_state, action, dynamics)
            total_reward += reward
            current_state = next_state
            states_over_time.append(current_state)
            if reward == -3:
                break


        accumulated_reward.append(total_reward)
        print(f"  Episode {episode + 1:2d} | Initial: {initial_state:4s} | Final: {current_state:4s} | Reward: {total_reward}")
    return states_over_time, accumulated_reward

policy_greedy   = {"high": "search", "low": "search"}
policy_balanced = {"high": "search", "low": "recharge"}

policies = [
    {"name": "Greedy (always search)",   "policy": policy_greedy,   "color": "red"},
    {"name": "Balanced (recharge low)",  "policy": policy_balanced, "color": "blue"},
]

For this task, the simulation was adapted to evaluate and compare two distinct policies. The key change introduced to `run_scenario` is the addition of a **policy parameter** — a dictionary that maps each state to a fixed action, replacing the random action selection used in Task 2. This means the robot no longer explores randomly but instead follows a deterministic rule at every step.

Additionally, a termination condition was introduced: if the robot is rescued (reward = -3), the current episode ends immediately via a `break` statement. This reflects the natural dynamics of the problem — a rescue resets the environment and marks the end of that interaction.

The two policies evaluated are:

**Policy 1 — Greedy (always search)**: The robot searches regardless of its battery level. This is the most aggressive strategy, maximizing the opportunity to collect cans but exposing the robot to a high risk of rescue when in the low state, since the probability of battery depletion is (1 - β).

**Policy 2 — Balanced (recharge when low)**: The robot searches when the battery is high but recharges immediately when it drops to low. This eliminates the risk of rescue entirely, at the cost of collecting no reward during recharge steps.

These two policies represent opposite ends of the risk-reward spectrum and provide a clear basis for comparison.

### 3.2 Simulation Results per Policy

In [10]:
policy_results = []
for p in policies:
    print(f"\n{'='*60}")
    print(f"  POLICY: {p['name']}")
    print(f"{'='*60}")
    states, rewards = run_scenario(alpha=0.8, beta=0.6, policy=p["policy"])
    policy_results.append({
        "name":    p["name"],
        "color":   p["color"],
        "states":  states,
        "rewards": rewards,
    })



  POLICY: Greedy (always search)

  SCENARIO | alpha=0.8, beta=0.6, r_search=5, r_wait=1
  Episode  1 | Initial: high | Final: high | Reward: 12
  Episode  2 | Initial: high | Final: high | Reward: 2
  Episode  3 | Initial: high | Final: low  | Reward: 50
  Episode  4 | Initial: high | Final: low  | Reward: 50
  Episode  5 | Initial: high | Final: high | Reward: 12
  Episode  6 | Initial: high | Final: high | Reward: 32
  Episode  7 | Initial: high | Final: high | Reward: 7
  Episode  8 | Initial: low  | Final: high | Reward: 7
  Episode  9 | Initial: high | Final: high | Reward: 27
  Episode 10 | Initial: high | Final: high | Reward: 50
  Episode 11 | Initial: high | Final: high | Reward: 17
  Episode 12 | Initial: low  | Final: high | Reward: 2
  Episode 13 | Initial: high | Final: low  | Reward: 50
  Episode 14 | Initial: low  | Final: high | Reward: 7
  Episode 15 | Initial: high | Final: high | Reward: 12
  Episode 16 | Initial: high | Final: low  | Reward: 50
  Episode 17 | Init

### 3.3 Policy Comparison

In [11]:
fig_policies = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Accumulated Reward over Episodes", "Reward Distribution"]
)

episodes = list(range(1, 51))

for result in policy_results:
    # Subplot 1: Recompensa acumulada
    fig_policies.add_trace(
        go.Scatter(
            x=episodes,
            y=list(np.cumsum(result["rewards"])),
            mode="lines",
            name=result["name"],
            line=dict(color=result["color"])
        ),
        row=1, col=1
    )
    # Subplot 2: Box plot
    fig_policies.add_trace(
        go.Box(
            y=result["rewards"],
            name=result["name"],
            marker_color=result["color"],
            boxmean=True
        ),
        row=1, col=2
    )

fig_policies.update_xaxes(title_text="Episode", col=1)
fig_policies.update_yaxes(title_text="Accumulated Reward", col=1)
fig_policies.update_yaxes(title_text="Reward per Episode", col=2)
fig_policies.update_layout(title_text="Policy Comparison | alpha=0.8, beta=0.6")
fig_policies.show()

### 3.4 Discussion

The simulation was run for both policies under the same scenario (α=0.8, β=0.6) across 50 episodes of up to 10 steps each.

The **Greedy policy** produced highly variable results, with rewards ranging from -3 to 50 across episodes. Episodes starting from the low state were particularly unpredictable — some ended immediately after a single step due to a rescue (reward = -3), while others accumulated significant rewards before depletion. This variability is a direct consequence of the (1 - β) = 0.4 probability of battery depletion when searching from the low state. A total of 9 out of 50 episodes ended in a rescue, consistently cutting those episodes short and penalizing the total reward.

The **Balanced policy** produced strikingly different results. Rewards ranged from 30 to 50, with no rescues across all 50 episodes. The recharge action guarantees a return to the high state whenever the battery drops to low, allowing the robot to search safely on every subsequent step. This results in a much tighter reward distribution and significantly higher per-episode rewards on average, as the robot spends more steps actively searching rather than recovering from or being cut short by a rescue.

The contrast between the two policies is already clear from the raw results: the Greedy policy trades consistency for occasional high-reward episodes, while the Balanced policy delivers stable and reliably high rewards across all episodes.